W&B 在工程上不仅仅是一个“画曲线的网站”，它是一套完整的 **MLOps（机器学习运维）基础设施**。它的核心体系由四个部分组成：**Runs（实验追踪）**、**Artifacts（产物与版本控制）**、**Sweeps（超参自动化搜索）** 和 **Reports（动态报告）**。

---

## 一、 W&B 的四大核心支柱

### 1. Runs（基础实验追踪）

这是最常用的部分。每一次你运行 `python train.py`，W&B 就会在云端创建一个独一无二的 **Run ID**。这个 Run 会死死盯住你的进程，实时抓取你的指标、系统硬件指标（DDP 模式下甚至能抓取每张卡的显存和通信状态）、控制台的标准输出（`stdout` / `stderr`）以及报错信息。

### 2. Artifacts（模型与数据集版本控制）

这是工业界极度看重的功能。传统的 `torch.save(model.state_dict(), 'best.pt')` 很容易重名或者搞混。
W&B 的 Artifacts 就像是**针对大数据、大模型的 Git**。它可以把你的数据集、训练好的模型 Checkpoint 自动上传到云端，并打上版本号（如 `v0`, `v1`, `latest`），甚至会**自动生成血统图（Lineage Graph）**，清晰地展示“这个模型是用哪一个数据集、哪一次 Run 训练出来的”。

### 3. Sweeps（自动化超参调优）

你不需要自己写 `for` 循环或者用 shell 脚本去试参数。Sweeps 允许你在云端定义一个搜索空间（比如学习率在 `1e-4` 到 `1e-2` 之间，优化器选 `AdamW` 或 `SGD`），W&B 会自动调度你的多台服务器，采用 **贝叶斯优化（Bayesian Optimization）**、**随机搜索** 或 **网格搜索** 自动帮你寻找最优超参组合，并带有**早停机制（Hyperband）**，发现模型跑崩了会自动掐断，省钱省时。

### 4. Reports（团队协作与技术沉淀）

它可以将你不同时期、不同项目跑出来的动态图表一键拖拽组合在一起。你可以在图表旁边写 Markdown Markdown 注释、插入数学公式，生成一个精美的在线网页。



在机器学习工程中，**W&B Artifacts（工件体系）** 承担着“版本控制系统”的角色。如果说 Git 是代码的版本管理工具，那么 Artifacts 就是**数据集、模型权重和大型二进制资产**的 Git。

通过将数据和模型组织为 Artifacts，W&B 能够自动追踪它们之间的依赖关系，形成一个清晰的**数据血缘图（Data Lineage）**。

---

## 一、 核心概念与工作流

一个完整的 Artifacts 闭环包含三个核心步骤：

1. **生产/构建（Create）：** 声明一个具有名称和类型的资产容器。
2. **挂载/打包（Add / New File）：** 将本地文件、目录或内存数据写入该容器。
3. **记录与使用（Log / Use）：** 将容器绑定到当前 Run，并跟踪其在整个流水线中的输入输出。

---

## 二、 核心 API 详解

### 1. 初始化工件容器：`wandb.Artifact`

这是构建任何工件的第一步，相当于在内存中开辟了一个虚拟的包裹。

```python
artifact = wandb.Artifact(name="mnist-origin", type="dataset", description="...", metadata={"size": 10000})

```

* **`name`**: 工件的名称。相同名称的工件多次提交，W&B 会自动将其递增为新版本（例如 `v0`, `v1`, `v2`）。
* **`type`**: 区分资产属性。常见的行业标准类型有：`"dataset"`（数据集）、`"model"`（模型权重）、`"code"`（源代码）。
* **`metadata`**: 一个键值对字典（Dict），用于存放无法直接写进文件的描述性元数据（如数据集大小、归一化参数等）。

---

### 2. 填充工件资产（三种常见操作）

根据数据源的不同，W&B 提供了不同的 API 将文件装进工件容器中：

* **操作 A：流式新建文件 (`artifact.new_file`)**
适用于直接在内存中将数据序列化到工件里（如 PyTorch 的 `torch.save`）。
```python
with artifact.new_file("data.pt", mode="wb") as file:
    torch.save((x, y), file)

```


* **操作 B：直接挂载本地单文件 (`artifact.add_file`)**
适用于模型训练完成后的权重文件（`.ckpt`, `.pth`）或单个代码脚本。
```python
artifact.add_file("./models/best_model.pth")

```


* **操作 C：直接挂载整个本地目录 (`artifact.add_dir`)**
适用于已经解压到本地的大型数据集文件夹。
```python
artifact.add_dir("./data/mnist_images/")

```



---

### 3. 提交与记录工件：`run.log_artifact` 或 `wandb.log_artifact`

将填充好资产的工件推送到 W&B 本地或云端仓库进行版本登记。

```python
run.log_artifact(artifact)

```

> 💡 **核心机制**：W&B 内部采用内容哈希（SHA-256）进行增量校验。如果你的数据集或模型文件没有发生任何内容改变，重新调用 `log_artifact` 不会占用额外的存储空间。

---

### 4. 声明和消费工件：`run.use_artifact` 与 `artifact.download`

在下游任务（如预处理、评估、推理）中，从仓库中获取特定版本的工件。

```python
# 1. 声明当前 Run 需要使用某一个特定版本的工件（latest 代表最新版，也可以指定 v1）
data_artifact = run.use_artifact('mnist-origin:latest')

# 2. 真正将数据下载到本地缓存盘，并返回本地路径字符串
local_dir = data_artifact.download()

```

---

## 三、 典型标准流水线工作流 (Data Pipeline)

在真实的生产流水线中，各个独立的脚本通常通过 `use_artifact` 和 `log_artifact` 相互串联，形成一个天然的管道：

```text
 ┌───────────────┐                  ┌───────────────────┐                  ┌───────────────┐
 │  Step 1: 加载 │                  │  Step 2: 预处理   │                  │  Step 3: 训练 │
 │               │                  │                   │                  │               │
 │  log_artifact │──mnist-origin───►│    use_artifact   │                  │  use_artifact │
 │ (mnist-origin)│     (latest)     │  (mnist-preprocess)◄─mnist-preprocess◄│(preprocess:lt)│
 └───────────────┘                  └───────────────────┘     (latest)     └───────┬───────┘
                                                                                   │
                                                                             log_artifact
                                                                             (best-model)
                                                                                   ▼
                                                                           [Artifact: model]

```

### 1. 数据导入节点 (Ingestion Run)

* **职责**：负责从原始服务器下载 MNIST，并创建原始基线。
* **流向**：`load()` ➔ `wandb.Artifact('mnist-origin')` ➔ `run.log_artifact()`。

### 2. 数据处理节点 (Preprocessing Run)

* **职责**：从上游拉取原始数据，进行归一化、扩维，并输出干净的数据供模型训练。
* **流向**：`run.use_artifact('mnist-origin:latest')` ➔ 读入内存 ➔ `preprocess()` ➔ `wandb.Artifact('mnist-preprocess')` ➔ `run.log_artifact()`。

### 3. 模型训练节点 (Training Run)

* **职责**：消费干净的数据集，训练出最终的模型权重工件。
* **流向**：`run.use_artifact('mnist-preprocess:latest')` ➔ 训练 ➔ `wandb.Artifact('cnn-model', type='model')` ➔ `run.log_artifact()`。

---

## 四、 实验复现与 Run 恢复 (`resume`)

如果你的实验由于断电中断，或者你想基于之前某次特定 Run 的基准继续训练并追加资产，可以利用 `resume` 机制精准衔接：

```python
# 通过具体的 project 和特定的 run_id 恢复现场，'must' 确保绝不创建新 Run
run = wandb.init(project='wandb_demo', id='model_run_id_1234', resume='must')

# 继续做你的增量实验，保存的新工件会自动挂载到这个旧 Run 的资产树下
arti_model = wandb.Artifact('cnn-enhanced', type='model')
arti_model.add_file(config.ckpt_path)
run.log_artifact(arti_model)

run.finish()

```

掌握了 Artifacts 的体系，整个团队无论是面对 T 级的数据集还是成百上千个模型 Checkpoint，都能轻松实现一键回溯与完美复现。


## 一、 两阶段代表的实际项目步骤

W&B Artifacts 的核心设计就是为了实现**数据与模型的解耦**。

* **阶段一 ＝ 数据流水线（Data Pipeline）**：
在实际项目中，这代表**数据工程师**（或定时脚本）从原始数据库拉取数据、清洗、标注，最终封装为**不可变的数据集资产（带版本号，如 `v1`）**。
* **阶段二 ＝ 模型训练与实验（Model Training）**：
这代表**算法工程师**进行实验。他们通过声明“消费”**阶段一产出的指定版本数据集（如 `v1`），进行训练，最终**“生产”出模型权重资产（如 `best_model:v1`）。

---

## 二、 这么做的核心好处（为什么不写在一个脚本里？）

W&B 强迫你把资产变成 Artifact 存入仓库，能降维打击三大痛点：

### 1. 绝对的实验复现（Data Lineage）

W&B 会自动在后台把“数据版本 ➔ 训练日志 ➔ 模型权重”连成一条线。
哪怕过了半年，你只要点开某个模型权重，就能一键回溯它当时是用**哪一个字节的数据**、哪一段代码训练出来的，彻底解决“模型再也跑不出来”的灾难。

### 2. 团队协作彻底解耦

数据团队尽可以去更新、发布 `dataset:v2` 和 `v3`；算法团队在跑长达几天的训练时，只要锁定 `use_artifact('dataset:v1')`，就不会受到任何上游原地修改数据的干扰。

### 3. 智能去重，节省 90% 空间

工件底层基于 **SHA-256 内容哈希** 校验。如果你的数据集没有变，或者每个 Epoch 存的模型权重内容几乎没变，无论你调用多少次 `log_artifact`，W&B 在底层**只存一份物理文件**，其余全是文件指针，极大节省硬盘。

In [10]:
# =====================================================================
# 0. 【环境守护补丁】强行保住旧版 pydantic_core 的兼容性，防止 Table/Artifact 冲突
# =====================================================================
import pydantic_core
import json
if not hasattr(pydantic_core, 'from_json'):
    pydantic_core.from_json = lambda s: json.loads(s if isinstance(s, str) else s.decode('utf-8'))

import os
import torch
import numpy as np
import wandb

# 创建一个本地虚拟的临时数据目录，模拟真实项目环境
os.makedirs("./local_disk_data", exist_ok=True)
os.makedirs("./local_disk_models", exist_ok=True)

# =====================================================================
# 阶段一：生产工件阶段（模拟从零生成原始数据集并打包上传）
# =====================================================================
def stage_1_produce_dataset():
    print("\n--- 🚀 启动阶段一：数据准备与工件生产 ---")
    
    # 初始化第一个 Run，任务类型标记为数据导入
    run = wandb.init(
        mode="offline",
        project="artifact_universe",
        name="data_ingestion_run",
        job_type="data-ingestion",
        dir="./wandb_offline_storage"
    )
    
    # 模拟在本地磁盘上生成了两个数据集文件
    train_x = np.random.rand(100, 1, 28, 28)
    train_y = np.random.randint(0, 10, size=(100,))
    torch.save((train_x, train_y), "./local_disk_data/raw_train.pt")
    
    with open("./local_disk_data/readme.txt", "w") as f:
        f.write("这是通过脚本生成的模拟原始 MNIST 数据集。")

    # 🏺 API 1: wandb.Artifact() -> 声明并实例化一个工件容器
    # 名字叫 'mnist-raw'，类型是 'dataset'，外加不可忽视的描述和元数据（便签纸）
    raw_data_artifact = wandb.Artifact(
        name="mnist-raw", 
        type="dataset",
        description="包含了初始生成的二进制训练集以及说明文档",
        metadata={
            "author": "zhangjinrui",
            "total_samples": 100,
            "data_type": "numpy_to_torch"
        }
    )
    
    # 🐣 API 2: artifact.add_file() -> 将本地单文件挂载到工件中
    raw_data_artifact.add_file("./local_disk_data/readme.txt")
    
    # 🐣 API 3: artifact.new_file() -> 绕过本地磁盘，在工件中流式创建并写入新文件
    # 这里我们把刚刚做好的 tensor 矩阵直接序列化进工件内部的 'train_tensor.pt'
    with raw_data_artifact.new_file("train_tensor.pt", mode="wb") as file:
        torch.save((train_x, train_y), file)
        
    # 🐣 API 4: artifact.add_dir() -> 把整个本地文件夹的所有内容一股脑打包打包挂载
    # [注]：这里为了演示完整API而保留，它会自动扫描整个 local_disk_data 目录
    raw_data_artifact.add_dir("./local_disk_data")

    # ✍️ API 5: run.log_artifact() -> 将工件正式提交登记，绑定到当前 Run
    run.log_artifact(raw_data_artifact)
    print("✨ 成功：原始数据集工件 'mnist-raw' 已封装并提交本地日志系统！")
    
    # 🏁 结束当前 Run
    run.finish()

In [11]:
# 依次执行两个阶段，形成标准的数据流水线（Data Pipeline）
stage_1_produce_dataset()


--- 🚀 启动阶段一：数据准备与工件生产 ---


/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/pydantic/main.py:301: UserWarning: Pydantic serializer warnings:
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


wandb: Adding directory to artifact (local_disk_data)... Done. 0.0s


✨ 成功：原始数据集工件 'mnist-raw' 已封装并提交本地日志系统！


In [ ]:
# =====================================================================
# 阶段二：消费工件与二次生产阶段
# =====================================================================
def stage_2_consume_and_train():
    print("\n--- 🚀 启动阶段二：消费数据工件 & 训练产出模型工件 ---")
    
    # 初始化第二个 Run
    run = wandb.init(
        mode="offline",
        project="artifact_universe",
        name="model_training_run",
        job_type="model-training",
        dir="./wandb_offline_storage"
    )

    # 📥 统一处理下载目录（离线走本地，在线走W&B）
    if run.offline:
        print("📢 检测到当前处于纯本地离线模式，自动重定向到物理磁盘数据源...")
        # 离线调试：直接赋给它阶段一落盘的本地物理文件夹路径
        download_dir = "./local_disk_data"
    else:
        print("🌐 检测到在线模式，正在从 W&B 云端服务器同步工件...")
        # 在线模式：才允许安全调用这两个 API
        raw_artifact_handle = run.use_artifact('mnist-raw:latest')
        download_dir = raw_artifact_handle.download()

    # 从确定的路径中，用标准 torch 重新加载数据进入内存
    target_data_path = os.path.join(download_dir, "raw_train.pt")
    x, y = torch.load(target_data_path)
    print(f"成功将消费的工件数据加载进训练内存，矩阵形状: {x.shape}")
    
    # --- 模拟网络训练 ---
    print("神经网络正在疯狂收敛中...")
    mock_model = torch.nn.Linear(10, 2)
    # 训练完后，在本地磁盘产生了一个权重文件
    torch.save(mock_model.state_dict(), "./local_disk_models/best_weight.pth")
    
    # 🏺 再次建立一个全新的模型类工件容器
    model_artifact = wandb.Artifact(
        name="cnn-model", 
        type="model",
        description="基于消费的原始数据训练出来的最优权重",
        metadata={"accuracy": 0.985, "epoch": 10}
    )
    
    # 将模型权重单文件装进包裹中
    model_artifact.add_file("./local_disk_models/best_weight.pth")
    
    # ✍️ 提交模型工件（离线模式下 log_artifact 是安全的）
    run.log_artifact(model_artifact)
    print("✨ 成功：产出的模型权重工件 'cnn-model' 已成功登记提交！")
    
    # 🏁 结束训练 Run
    run.finish()

In [13]:
stage_2_consume_and_train()


--- 🚀 启动阶段二：消费数据工件 & 训练产出模型工件 ---


/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/pydantic/main.py:301: UserWarning: Pydantic serializer warnings:
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


📢 检测到当前处于纯本地离线模式，自动重定向到物理磁盘数据源...
成功将消费的工件数据加载进训练内存，矩阵形状: (100, 1, 28, 28)
神经网络正在疯狂收敛中...
✨ 成功：产出的模型权重工件 'cnn-model' 已成功登记提交！



## 一、 核心 API：外部引用（Reference）

在**在线联机模式**下，面对几十 GB 的大数据，我们不要用 `add_dir` 上传，而是改用 **`add_reference("file://本地绝对路径")`**。

### 它的骚操作在于：

W&B **不上传任何实际数据文件**（流量、云端存储体积均为 0）。它只做两件事：

1. 扫描你本地大文件夹里所有文件的**名字、大小和 SHA-256 哈希值**。
2. 把这些哈希指纹和本地路径组成一个精简的账本，打包上传到云端，命名为 `huge-dataset:v1`。

---

## 二、 本地数据变了，W&B 怎么识别和锁死？

到了阶段二训练时，你依然在代码里正常调用官方标准的 `run.use_artifact('huge-dataset:latest')`。

W&B 依靠“对账机制”铁面无私地替你把关：

1. **文件被删改（哈希对不上）**：
如果阶段一到阶段二之间，你本地的超大数据集不小心被删了 10 张图，或者改了某些内容。当你再次触发 `dataset.download()` 时，W&B 的本地客户端会自动重新计算一遍本地数据的哈希值，跟云端账本里的 `v1` 密码本一碰：**指纹对不上，当场直接报错中断**，防止你用脏数据跑了错误的实验。
2. **数据主动更新（版本自动升级）**：
如果你主动更新了本地大数据，并重新跑了一遍阶段一的注册代码，W&B 发现指纹变了，会在云端**自动生成 `huge-dataset:v2**`。
此时你的训练脚本只要锁定 `use_artifact('huge-dataset:v1')` 还是 `v2`，就能在不移动数据的前提下，精准控制训练所用数据的版本。

## 📌 总结一句话

这就是所谓的 **“数据肉体留本地，哈希灵魂上云端”**。你一分钱的网络流量都不花，但 W&B 用**哈希密码本**帮你把本地几十个 GB 的大数据死死锁在了云端的版本树上！

In [14]:
import os
import numpy as np
import torch
import wandb

def stage_1_register_large_dataset():
    print("\n--- 🌐 启动阶段一：大数据本地锁死与云端指纹登记 ---")
    
    # 1. 开启标准的【在线联机】模式
    run = wandb.init(
        mode="online", 
        entity="Jerry-Auto",
        project="learn_wandb",
        name="huge_data_registration",
        job_type="data-registration"
    )
    
    # 模拟在本地磁盘生成了一个 50GB 的超大原始数据集（肉体留本地）
    os.makedirs("./my_large_disk_data", exist_ok=True)
    big_matrix = np.random.rand(1000, 1, 28, 28)
    torch.save(big_matrix, "./my_large_disk_data/huge_train_matrix.pt")
    
    # 🏺 2. 声明工件容器
    large_dataset_artifact = wandb.Artifact(
        name="huge-mnist-dataset", 
        type="dataset",
        description="存放在本地大硬盘上的超大型数据集，仅在云端进行指纹版本控制"
    )
    
    # 获取本地大文件夹的绝对物理路径
    local_absolute_path = os.path.abspath("./my_large_disk_data")
    
    # 🚀 【核心 API】: add_reference()
    # 重点：传入 file:// 协议开头的绝对路径。W&B 此时只会扫描并计算该目录下所有文件的 SHA-256 哈希值
    large_dataset_artifact.add_reference(f"file://{local_absolute_path}")
    
    # ✍️ 3. 登记到云端（上传的只是几 KB 的哈希清单，瞬间完成，不耗流量）
    run.log_artifact(large_dataset_artifact)
    print(f"✨ 成功：本地大数据指纹已登记为 'huge-mnist-dataset:latest'！")
    
    run.finish()

In [15]:
stage_1_register_large_dataset()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zhangjinrui/.netrc.



--- 🌐 启动阶段一：大数据本地锁死与云端指纹登记 ---


wandb: Currently logged in as: 1763287396 (Jerry-Auto) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/pydantic/main.py:301: UserWarning: Pydantic serializer warnings:
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


wandb: Generating checksum for up to 10000000 files in '/home/zhangjinrui/my_program_code/Data_Processing/python/torch/15_advance_training/my_large_disk_data'... Done. 0.0s


✨ 成功：本地大数据指纹已登记为 'huge-mnist-dataset:latest'！


In [ ]:
import os
import torch
import wandb

def stage_2_train_with_safeguard():
    print("\n--- 🌐 启动阶段二：在线拉取账本 & 本地哈希对账训练 ---")
    
    # 1. 开启【在线联机】模式
    run = wandb.init(
        mode="online",
        entity="Jerry-Auto",
        project="learn_wandb",
        name="model_training_with_large_data",
        job_type="model-training"
    )

    # ✔️ 2. 向上游仓库声明：“我要用这个超大数据集的最新版”
    # 此时云端会把当初登记的虚拟账本发给本地客户端
    dataset_artifact = run.use_artifact('huge-mnist-dataset:latest')
    
    # 📥 3. 执行 download() 触发对账机制
    # 【机制】：因为这是引用工件，W&B 绝不会从网络下载 50GB 数据！
    # 它会自动扫描当初登记的本地绝对路径，并现场重新计算本地文件的哈希值。
    # 💡 如果你偷偷改了本地数据，这一步会直接抛出【哈希不匹配异常】强行拦截！
    try:
        download_dir = dataset_artifact.download()
        print(f"✅ 对账成功！数据未被篡改。安全对接本地大数据路径: {download_dir}")
    except Exception as e:
        print(f"❌ 警告：本地大数据已被篡改或移动！与云端锁死的哈希版本不一致！\n错误信息: {e}")
        run.finish()
        return

    # 4. 确认安全后，正常从路径中加载大文件进行训练
    target_data_path = os.path.join(download_dir, "huge_train_matrix.pt")
    big_matrix = torch.load(target_data_path)
    print(f"数据矩阵成功加载，形状为: {big_matrix.shape}")
    
    # --- 模拟网络训练与模型工件产出 ---
    mock_model = torch.nn.Linear(10, 2)
    os.makedirs("./my_large_disk_models", exist_ok=True)
    torch.save(mock_model.state_dict(), "./my_large_disk_models/best_large_model.pth")
    
    model_artifact = wandb.Artifact(name="large-trained-model", type="model")
    model_artifact.add_file("./my_large_disk_models/best_large_model.pth")
    
    # 提交模型，此时在云端就会自动生成【huge-mnist-dataset】->【当前Run】->【large-trained-model】的震撼血缘图
    run.log_artifact(model_artifact)
    print("✨ 成功：模型工件已提交，数据血缘完全闭环！")
    
    run.finish()

In [17]:
stage_2_train_with_safeguard()


--- 🌐 启动阶段二：在线拉取账本 & 本地哈希对账训练 ---


/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/pydantic/main.py:301: UserWarning: Pydantic serializer warnings:
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


wandb:   1 of 1 files downloaded.  


✅ 对账成功！数据未被篡改。安全对接本地大数据路径: /home/zhangjinrui/my_program_code/Data_Processing/python/torch/15_advance_training/artifacts/huge-mnist-dataset:v0
数据矩阵成功加载，形状为: (1000, 1, 28, 28)
✨ 成功：模型工件已提交，数据血缘完全闭环！


进入下一个核心知识点：**`wandb.Table`（交互式 Case 分析）**。

在实际的深度学习项目中，只看 Accuracy 或 Loss 曲线（宏观指标）是远远不够的。当模型遇到瓶颈时，我们必须深入到**微观数据**中去，看看模型到底把哪些样本预测错了（Bad Cases），又把哪些样本预测对了（Good Cases）。这也就是俗称的 **“抓 Case”**。

W&B 提供的 `wandb.Table` 就像是一个**网页版的超级 Excel**，能把图片、文本、标签、预测值等信息揉进一张表里，让你在网页端直接进行过滤、排序和分组分析。

---

## 一、 核心概念与工作流

`wandb.Table` 的本质是一个**结构化的二维数据表**。它的生命周期同样遵循：

1. **创建表结构**：声明表头（Columns）。
2. **填充数据行**：按行追加样本信息（可以包含纯文本、数字，以及 `wandb.Image` 多媒体资产）。
3. **上传与交互**：通过 `wandb.log()` 提交，然后在前端看板上开启“神仙级”的数据洞察。

---

## 二、 核心 API 详解

### 1. 初始化表格：`wandb.Table`

在内存中定义一张空白表，必须明确指定有哪些列。

```python
table = wandb.Table(columns=['Image', 'GroundTruth', 'Prediction'])

```

* **`columns`**: 字符串列表（List[str]），代表表格的表头。定义好后，后续添加的每一行数据，其元素个数和顺序必须与表头严格对齐。

### 2. 封装多媒体资产：`wandb.Image`

表格不仅能存数字和字符串，最强大的地方在于能存多媒体。如果想把图片塞进表格里，必须先用 `wandb.Image` 包装：

```python
# 将 PIL Image 对象或 NumPy 数组包装为 W&B 的多媒体格式
input_img = wandb.Image(pil_image_object)

```

### 3. 追加数据行：`table.add_data`

采用“来一个，塞一个”的行级追加模式。

```python
table.add_data(input_img, label, y_pred)

```

* **注意**：每次调用的入参，必须对应初始化的 `columns`。例如这里传入的三个变量分别对应 `['Image', 'GroundTruth', 'Prediction']`。

### 4. 提交表格：`wandb.log`

将整张表作为字典的 Value，打包提交。

```python
wandb.log({'bad_cases': table})

```

---

## 三、 实际项目中的适用流程

在标准的工业级模型诊断（Model Diagnostics）中，Case 分析通常分为以下三步：

### 🎬 步骤 1：模型推理与分流（筛选 Good/Bad Case）

让模型在验证集或测试集上跑一遍推理，通过一个简单的 `if-else` 条件判断：

* 如果 `y_pred == label`：归入 **Good Cases** 表（研究模型已经具备了什么能力）。
* 如果 `y_pred != label`：归入 **Bad Cases** 表（研究模型有什么致命缺陷，重点攻克）。

### 🎬 步骤 2：多媒体序列化落盘

将张量（Tensor）图片转换回肉眼可看的格式（如通过 Matplotlib 转成黑白/彩色图，再转为 PIL Image），然后封装进 `wandb.Table`，打完收工，上传云端。

### 🎬 步骤 3：前端看板交互分析（重点）

这也是为什么不用本地写 `plt.show()` 的原因。数据传到 W&B Dashboard 后，你可以像用 Excel 一样在网页上进行高级交互：

* 🔍 **Filter（过滤）**：输入表达式（例如 `Prediction == 7`），一键找出所有被模型错认成“7”的图片。这时候你一眼就能看出，是不是因为某些数字写得太歪，导致模型把“1”和“7”搞混了。
* 📊 **Group By（分组）**：按 `GroundTruth` 分组，看看哪一个类别的错题最多，精准定位模型的弱点。
* 🔢 **Sort（排序）**：在有置信度（Confidence）的情况下，按置信度从大到小排序，抓出那些“模型极其自信，但依然错得离谱”的**恶性 Bad Case**。

---

## 四、 核心好处提炼

1. **摆脱本地 GUI 束缚**：再也不用在 Jupyter Notebook 或服务器上写笨重的 `plt.subplot(3,3,i+1)`。一旦样本量过千，本地绘图直接卡死，而网页版 Table 轻松承载万级数据。
2. **多维度动态探索**：本地画出来的图是静态的死图。`wandb.Table` 支持在网页上随时改变过滤条件，算法团队的所有人都能登录同一个网页，各取所需地去筛查自己负责的类别的 Bad Case，协同效率极高。

In [19]:
# =====================================================================
# 0. 【环境守护补丁】强行保住旧版 pydantic_core 的兼容性
# =====================================================================
import pydantic_core
import json
if not hasattr(pydantic_core, 'from_json'):
    pydantic_core.from_json = lambda s: json.loads(s if isinstance(s, str) else s.decode('utf-8'))

import numpy as np
import wandb

def run_online_case_analysis():
    print("\n--- 🌐 启动联机模式：纯内存数据直传云端表格 ---")
    
    # 1. ⚡ 核心改动：切换为 "online"，让数据实时同步上云
    run = wandb.init(
        mode="online",
        entity="Jerry-Auto",
        project="learn_wandb",
        name="live_table_test"
    )

    # 📊 2. 创建空白表格，定义好 Excel 表头
    case_table = wandb.Table(columns=['Image', 'GroundTruth', 'Prediction', 'Status'])

    print("🔍 正在纯内存模拟 5 张图片的推理与分流...")

    for i in range(5):
        # 🎲 模拟 A：内存随机生成一张 28x28 的假灰度噪点图
        fake_img_np = np.random.rand(28, 28)
        
        # 🎲 模拟 B：随机生成标签和预测值（制造对错分流）
        label = np.random.randint(0, 10)
        y_pred = np.random.randint(0, 10)
        status = "Good" if y_pred == label else "Bad"

        # 🚀 包装多媒体图像资产
        wandb_img = wandb.Image(fake_img_np, caption=f"Sample_{i}")

        # 🚀 按行塞进表格
        case_table.add_data(wandb_img, label, y_pred, status)

    # ✍ =====================================================================
    # 3. 实时发送！一眨眼数据就会出现在你的 W&B Dashboard 网页中
    # =====================================================================
    run.log({"live_case_analyzer": case_table})
    print("\n✨ 成功：数据已实时推送到云端！请点击上方控制台输出的 URL 链接查看。")
    
    run.finish()
    

In [20]:
run_online_case_analysis()


--- 🌐 启动联机模式：纯内存数据直传云端表格 ---


/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/pydantic/main.py:301: UserWarning: Pydantic serializer warnings:
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  Expected `list[str]` but got `tuple` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


🔍 正在纯内存模拟 5 张图片的推理与分流...

✨ 成功：数据已实时推送到云端！请点击上方控制台输出的 URL 链接查看。
